# Benchmark de Previsão Integrada Multinó em Redes LoRaWAN

Este notebook avalia o trade-off entre **modelos dedicados uninó (*Single-Node*)** vs. **modelos integrados multinó (*Multi-Node / Joint*)** para previsão de RSSI em redes LoRaWAN.

### Melhorias e Correções Implementadas:
1. **Pipeline causal:** separação temporal 70/15/15, scaler ajustado no treino e descarte de janelas incompletas ou não horárias, sem imputação.
2. **Eliminação do Teacher Forcing no Teste:** Modelos geram horizontes multi-passo sem vazamento do gabarito futuro.
3. **TCN com Convoluções Causais Dilatadas Reais:** Fatores de dilatação exponenciais ($d=1, 2, 4, 8, 16$) com amplo campo receptivo.
4. **Métricas Físicas de Telecomunicações:** Erro medido estritamente em **MAE (dBm)** e **RMSE (dBm)**, descartando o MAPE sobre grandezas logarítmicas.

In [ ]:
import sys
import os
import json
import matplotlib.pyplot as plt

from src.data_loader import get_prepared_datasets, TARGET_COLS, EXOGENOUS_COLS
from src.evaluation import run_full_benchmark
from src.plotting import plot_predictions_comparison, plot_benchmark_metrics

print('Ambiente e módulos carregados com sucesso!')

## 1. Carregamento e Inspeção dos Dados Sem Vazamento

In [ ]:
data = get_prepared_datasets(
    file_path='data/combined_hourly_data.csv',
    seq_length=24,
    pred_length=6
)
print(f"Treino: {data['train'][0].shape} -> {data['train'][1].shape}")
print(f"Validação: {data['val'][0].shape} -> {data['val'][1].shape}")
print(f"Teste: {data['test'][0].shape} -> {data['test'][1].shape}")
print('Nós monitorados:', data['target_names'])
print('Variáveis do modelo:', data['feature_names'])

## 2. Execução do Benchmark Completo

Executamos o comparador com:
- **Baselines:** Persistence (Naive) e Moving Average.
- **Uninó:** Ensemble de 8 LSTMs dedicadas.
- **Multinó Integrado:** Multi-Node Seq2Seq + Attention e Multi-Node Dilated TCN.
- **Linear moderno:** NLinear (AAAI 2023).
- **Espaço-temporal:** STGNN com grafo físico GPS e adjacência adaptativa.

In [ ]:
HORIZON = 6  # 6 horas à frente (médio prazo)
results = run_full_benchmark(
    data_file='data/combined_hourly_data.csv',
    seq_length=24,
    pred_length=HORIZON,
    epochs=25,
    output_dir='benchmark_results'
)

## 3. Análise dos Resultados e Trade-off de Integração

In [ ]:
print('='*75)
header = f"{'Modelo':<20} | {'MAE (dBm)':<10} | {'RMSE (dBm)':<10} | {'Params':<9} | {'Latency':<8}"
print(header)
print('-'*75)
for m in sorted(results.keys(), key=lambda k: results[k]['metrics']['global']['mae_dbm']):
    mae = results[m]['metrics']['global']['mae_dbm']
    rmse = results[m]['metrics']['global']['rmse_dbm']
    params = results[m].get('parameters', 0)
    lat = results[m].get('latency_ms', 0.0)
    p_str = f"{params:,}" if params > 0 else '-'
    l_str = f"{lat:.2f}ms" if lat > 0 else '-'
    print(f"{m:<20} | {mae:<10.4f} | {rmse:<10.4f} | {p_str:<9} | {l_str:<8}")
print('='*75)

## 4. Visualização dos Gráficos Gerados

In [ ]:
from IPython.display import Image, display
display(Image(filename=f'benchmark_results/metrics_comparison_H{HORIZON}.png'))
display(Image(filename=f'benchmark_results/predictions_comparison_H{HORIZON}.png'))